# GENCO's response to realistic changes

Where GENCO is wrong, is it wrong in a systematic way? This notebook perturbs ordinary N-2 operating points and compares
GENCO's response with the AC power-flow Jacobian at the true solution (checked against re-solving the power flow):

- (a) one reactive load at a time, ±5 Mvar, at increasing distance from the bus being watched;
- (b) every load growing together by ±2 % at constant power factor, the slack taking the extra power;
- (c) the voltage set-point of the nearest generators, ±0.005 p.u.

Runtime: T4 GPU, about 1.2 GB of downloads, 20 to 30 minutes. Output: `load_response.zip`.

## Settings

In [ ]:
REPO_ID = "gridfm/genco-pf-contingency"
HF_REVISION = "5c625b00f5b79c8a7d4d27ecc133d4ecdf573af7"       # the revision every run used (unchanged since 1 October 2026)
GRAPHKIT_COMMIT = "95b90eecc7f5c2c86a9dd776f0c3f40434def22f"   # the commit every run used (branch head from 1 to 14 October 2026)
LOCAL_DIR = "genco-pf-contingency"
OUT_DIR = "load_response"
GRAPHKIT_DIR = "gridfm-graphkit"
GRAPHKIT_BRANCH = "genco-paper-repro"
NETWORK = "Texas2k_case1_2016summerpeak"
MODEL_DIR = "model/training_bs_16"

LEVELS = [2, 10, 20]          # outage levels
PARTS = [0, 1]                # data blocks per level
N_PER_LEVEL = 300             # scenarios GENCO is run on, per level
COMPARE_LEVEL = 20            # reproduction check against the released predictions
BATCH = 8                     # forward passes
BATCH_GRAD = 4                # gradient passes (smaller: they keep the whole computation in memory)
HARD_ERR = 0.01               # p.u.: a bus is badly predicted, and a scenario hard, above this voltage error
N_GRAD_HARD = 30              # per level: hard scenarios whose sensitivities are computed
N_GRAD_NORMAL = 30            # per level: normal scenarios
WATCH_BUS = 1797
N_TARGET_MISSED = 10          # target buses GENCO most often gets wrong
N_TARGET_RANDOM = 20          # random target buses
J2_HARD = 3                   # per level: hard scenarios for tests (a) and (c)
J2_NORMAL = 3                 # per level: normal scenarios for tests (a) and (c)
DQ_MVAR = 5.0                 # (a) one reactive load, +/- this many Mvar
GROWTH = 0.02                 # (b) all loads, +/- 2 %
DV_PU = 0.005                 # (c) generator voltage set-point, +/- this many p.u.
N_NEAR, N_MID, N_FAR = 3, 3, 3   # (a) loads per target in each distance class
N_GENQ = 2                    # (a) generator-bus loads per target
N_SET = 3                     # (c) nearest generators per target
BATCH_PERT = 16
MAX_DOWNLOAD_GB = 4
SEED = 0

In [ ]:
# [GPU] install and import

import importlib.util, os, subprocess, sys, time


def run(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise RuntimeError(f"Command failed: {cmd}")


import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
if not torch.cuda.is_available():
    print("WARNING: no GPU. Use Runtime > Change runtime type > T4 GPU, then Runtime > Run all again.\n"
          "The notebook also works on CPU, but the timing results would not be meaningful.")

if not os.path.isdir(GRAPHKIT_DIR):
    run(f"git clone -q -b {GRAPHKIT_BRANCH} https://github.com/gridfm/gridfm-graphkit.git {GRAPHKIT_DIR}")
    run(f"git -C {GRAPHKIT_DIR} checkout -q {GRAPHKIT_COMMIT}")   # pinned: the commit every run used
need = {"torch_geometric": "torch_geometric", "lightning": "lightning", "pytorch_lightning": "pytorch-lightning",
        "huggingface_hub": "huggingface_hub", "pyarrow": "pyarrow", "seaborn": "seaborn", "sklearn": "scikit-learn"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    run(f"{sys.executable} -m pip install -q " + " ".join(missing))

import torch_geometric   # imported before the replacement below, so PyTorch Geometric keeps its own native code paths
USED_SHIM = importlib.util.find_spec("torch_scatter") is None
if USED_SHIM:
    shim = os.path.abspath(os.path.join("shims", "torch_scatter"))
    os.makedirs(shim, exist_ok=True)
    with open(os.path.join(shim, "__init__.py"), "w") as f:
        f.write('"""Small pure-PyTorch stand-in for torch_scatter (only the functions gridfm-graphkit uses)."""\nimport torch\n\n__version__ = "0.0.0-shim"\n\n\ndef _prep(src, index, dim):\n    dim = dim if dim >= 0 else src.dim() + dim\n    if index.dim() != src.dim():\n        shape = [1] * src.dim()\n        shape[dim] = -1\n        index = index.view(shape).expand_as(src)\n    return index, dim\n\n\ndef _size(src, index, dim, dim_size):\n    size = list(src.shape)\n    if dim_size is None:\n        dim_size = int(index.max()) + 1 if index.numel() > 0 else 0\n    size[dim] = dim_size\n    return size\n\n\ndef scatter_add(src, index, dim=-1, out=None, dim_size=None):\n    index, dim = _prep(src, index, dim)\n    if out is None:\n        out = torch.zeros(_size(src, index, dim, dim_size), dtype=src.dtype, device=src.device)\n    return out.scatter_add_(dim, index, src)\n\n\nscatter_sum = scatter_add\n\n\ndef scatter_mean(src, index, dim=-1, out=None, dim_size=None):\n    total = scatter_add(src, index, dim, None, dim_size)\n    idx, d = _prep(src, index, dim)\n    count = torch.zeros_like(total).scatter_add_(d, idx, torch.ones_like(src)).clamp_(min=1)\n    return total / count\n\n\ndef scatter_max(src, index, dim=-1, out=None, dim_size=None):\n    idx, d = _prep(src, index, dim)\n    res = torch.full(_size(src, idx, d, dim_size), float("-inf"), dtype=src.dtype, device=src.device)\n    res = res.scatter_reduce(d, idx, src, reduce="amax", include_self=True)\n    res[torch.isinf(res)] = 0\n    return res, None\n\n\ndef scatter(src, index, dim=-1, out=None, dim_size=None, reduce="sum"):\n    if reduce in ("sum", "add"):\n        return scatter_add(src, index, dim, out, dim_size)\n    if reduce == "mean":\n        return scatter_mean(src, index, dim, out, dim_size)\n    if reduce == "max":\n        return scatter_max(src, index, dim, out, dim_size)[0]\n    raise NotImplementedError(reduce)\n')
    sys.path.insert(0, os.path.abspath("shims"))
    print("torch_scatter is not installed: using a small built-in replacement")
sys.path.insert(0, os.path.abspath(GRAPHKIT_DIR))

import gridfm_graphkit
from gridfm_graphkit.io.param_handler import NestedNamespace, load_normalizer, get_task_transforms, get_task
from gridfm_graphkit.training.callbacks import adapt_state_dict_for_model
from gridfm_graphkit.datasets.hetero_preprocess import build_hetero_data_for_scenario, merge_agg_gen_into_bus
from gridfm_graphkit.datasets.globals import VM_OUT, VA_OUT
from torch_geometric.loader import DataLoader
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch_geometric", torch_geometric.__version__, "| gridfm-graphkit from", os.path.abspath(GRAPHKIT_DIR), "| device:", DEVICE)

In [ ]:
# ===================== core functions =====================
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import scipy.sparse as sp
from scipy.sparse.linalg import splu

SN_MVA = 100.0
BUS_COLS = ["scenario", "bus", "Pd", "Qd", "Pg", "Qg", "Vm", "Va", "PQ", "PV", "REF",
            "min_vm_pu", "max_vm_pu", "GS", "BS", "Va_dc", "vn_kv"]
BR_COLS = ["scenario", "idx", "from_bus", "to_bus", "Yff_r", "Yff_i", "Yft_r", "Yft_i", "Ytf_r", "Ytf_i",
           "Ytt_r", "Ytt_i", "rate_a", "br_status", "pf_dc", "pt_dc"]
PRED_ALIASES = {"vm": ["vm_pu", "Vm_pred", "VM_pred", "vm_pred"], "va": ["va", "Va_pred", "VA_pred", "va_pred"]}


# ---------------------------------------------------------------- file helpers
def find_local(local_dir, k, table):
    base = Path(local_dir) / "data" / f"test_{k}"
    hits = sorted(base.rglob(f"{table}.parquet")) if base.exists() else []
    if not hits:
        raise FileNotFoundError(f"No {table}.parquet under {base}")
    return hits[0]


def find_pred(local_dir, k):
    base = Path(local_dir) / "mlflow" / "eval" / f"test_{k}"
    hits = sorted(base.rglob("*predictions.parquet")) if base.exists() else []
    hits = [h for h in hits if not re.search(r"_(gen|branch)_predictions\.parquet$", h.name)]
    if not hits:
        raise FileNotFoundError(f"No predictions.parquet under {base}")
    return hits[0]


def _dataset(path):
    return ds.dataset(str(path), format="parquet", partitioning="hive")


def read_cols(path, wanted, scen=None):
    dset = _dataset(path)
    cols = [c for c in wanted if c in dset.schema.names]
    flt = None
    if scen is not None:
        values = pa.array(np.asarray(scen, dtype=np.int64)).cast(dset.schema.field("scenario").type)
        flt = ds.field("scenario").isin(values)
    return dset.to_table(columns=cols, filter=flt).to_pandas()


def scenario_ids(path):
    return np.unique(_dataset(path).to_table(columns=["scenario"]).column("scenario").to_numpy().astype(np.int64))


# ---------------------------------------------------------------- one chunk of scenarios
def load_chunk(bus_path, br_path, pred_chunk, scen):
    """Arrays for a chunk of scenarios. pred_chunk: DataFrame with scenario, bus, vm, va (already filtered)."""
    bus = read_cols(bus_path, BUS_COLS, scen=scen)
    bus["scenario"] = bus["scenario"].astype(np.int64)
    bus = bus.sort_values(["scenario", "bus"], kind="stable").reset_index(drop=True)
    pred = pred_chunk.sort_values(["scenario", "bus"], kind="stable").reset_index(drop=True)
    S = len(scen)
    nb = len(bus) // S
    if len(bus) != S * nb or len(pred) != S * nb:
        raise RuntimeError(f"rows do not match: bus {len(bus)}, predictions {len(pred)}, {S} scenarios")
    ids = np.broadcast_to(np.arange(nb), (S, nb))
    if not (np.array_equal(bus["bus"].to_numpy().reshape(S, nb), ids) and np.array_equal(pred["bus"].to_numpy().reshape(S, nb), ids)):
        raise RuntimeError("bus numbers are not 0..nb-1 in every scenario")
    A = {c: bus[c].to_numpy(np.float64).reshape(S, nb) for c in BUS_COLS[2:] if c in bus.columns}
    A["PQ"], A["PV"], A["REF"] = A["PQ"] > 0.5, A["PV"] > 0.5, A["REF"] > 0.5
    A["Vm_g"] = pred["vm"].to_numpy(np.float64).reshape(S, nb)
    va = pred["va"].to_numpy(np.float64).reshape(S, nb)
    rad_err = np.nanmedian(np.abs(np.degrees(va) - A["Va"]))
    deg_err = np.nanmedian(np.abs(va - A["Va"]))
    A["va_unit"] = "rad" if rad_err <= deg_err else "deg"
    A["Va_g"] = np.degrees(va) if A["va_unit"] == "rad" else va
    br = read_cols(br_path, BR_COLS, scen=scen)
    A["sc"] = np.searchsorted(scen, br["scenario"].to_numpy().astype(np.int64))
    A["line"] = (br["idx"].to_numpy().astype(np.int64) if "idx" in br.columns
                 else br.groupby("scenario").cumcount().to_numpy().astype(np.int64))
    A["f"] = br["from_bus"].to_numpy().astype(np.int64)
    A["t"] = br["to_bus"].to_numpy().astype(np.int64)
    for y in ["Yff", "Yft", "Ytf", "Ytt"]:
        A[y] = br[y + "_r"].to_numpy(np.float64) + 1j * br[y + "_i"].to_numpy(np.float64)
    A["rate"] = br["rate_a"].to_numpy(np.float64)
    A["on"] = (br["br_status"].to_numpy(np.float64) > 0.5) if "br_status" in br.columns else (np.abs(A["Yff"]) > 0)
    pdc = br["pf_dc"].to_numpy(np.float64) if "pf_dc" in br.columns else np.full(len(br), np.nan)
    ptdc = br["pt_dc"].to_numpy(np.float64) if "pt_dc" in br.columns else pdc
    A["P_dc_abs"] = np.maximum(np.abs(pdc), np.abs(ptdc))
    A["S"], A["nb"], A["scen"] = S, nb, np.asarray(scen)
    return A


def branch_flows(A, V):
    sc, f, t = A["sc"], A["f"], A["t"]
    Vf, Vt = V[sc, f], V[sc, t]
    Sf = Vf * np.conj(A["Yff"] * Vf + A["Yft"] * Vt) * SN_MVA
    St = Vt * np.conj(A["Ytt"] * Vt + A["Ytf"] * Vf) * SN_MVA
    return Sf, St


def bus_mismatch(A, Vm, Sf, St):
    S, nb, sc, f, t = A["S"], A["nb"], A["sc"], A["f"], A["t"]
    add = lambda v, at: np.bincount(sc * nb + at, weights=v, minlength=S * nb).reshape(S, nb)
    P = (A["Pg"] - A["Pd"]) - (add(Sf.real, f) + add(St.real, t)) - A["GS"] * Vm**2 * SN_MVA
    Q = (A["Qg"] - A["Qd"]) - (add(Sf.imag, f) + add(St.imag, t)) + A["BS"] * Vm**2 * SN_MVA
    return P, Q


def _scen_max(values, sc, mask, S):
    out = np.full(S, np.nan)
    v = np.where(mask, values, -np.inf)
    tmp = np.full(S, -np.inf); np.maximum.at(tmp, sc, v)
    out[np.isfinite(tmp)] = tmp[np.isfinite(tmp)]
    return out


def chunk_tables(A, near_line=0.4, near_v=0.08):
    """Compact line-level and bus-level tables plus per-scenario signals for one chunk."""
    S, nb = A["S"], A["nb"]
    V_t = A["Vm"] * np.exp(1j * np.radians(A["Va"]))
    V_g = A["Vm_g"] * np.exp(1j * np.radians(A["Va_g"]))
    V_d = np.exp(1j * np.radians(A["Va_dc"]))
    Sf_t, St_t = branch_flows(A, V_t)
    Sf_g, St_g = branch_flows(A, V_g)
    Sf_d, St_d = branch_flows(A, V_d)
    P_t, _ = bus_mismatch(A, A["Vm"], Sf_t, St_t)
    P_g, Q_g = bus_mismatch(A, A["Vm_g"], Sf_g, St_g)
    resP = np.where(A["REF"], np.nan, np.abs(P_g))
    with np.errstate(all="ignore"):
        r_p99 = np.nanpercentile(resP, 99, axis=1)
    rated = A["on"] & (A["rate"] > 0)
    rate = np.where(rated, A["rate"], np.nan)
    ld = lambda Sf, St: np.maximum(np.abs(Sf), np.abs(St)) / rate
    ld_t, ld_g, ld_dS = ld(Sf_t, St_t), ld(Sf_g, St_g), ld(Sf_d, St_d)
    ld_dP = A["P_dc_abs"] / rate
    dc_failed = np.isnan(A["Va_dc"]).any(axis=1)
    keep = rated & ((ld_t >= near_line) | (ld_g >= near_line) | (np.nan_to_num(ld_dS) >= near_line) | (np.nan_to_num(ld_dP) >= near_line))
    lines = pd.DataFrame({
        "sidx": A["sc"][keep].astype(np.int32), "line": A["line"][keep].astype(np.int32),
        "ld_true": ld_t[keep].astype(np.float32), "ld_genco": ld_g[keep].astype(np.float32),
        "ld_dc_paper": ld_dS[keep].astype(np.float32), "ld_dc_p": ld_dP[keep].astype(np.float32),
    })
    lines.loc[dc_failed[lines.sidx.to_numpy()], ["ld_dc_paper", "ld_dc_p"]] = np.nan
    # PQ buses close to a voltage limit (truth or GENCO)
    PQ, vmin, vmax = A["PQ"], A["min_vm_pu"], A["max_vm_pu"]
    dist_t = np.minimum(A["Vm"] - vmin, vmax - A["Vm"])
    dist_g = np.minimum(A["Vm_g"] - vmin, vmax - A["Vm_g"])
    kb = PQ & ((dist_t <= near_v) | (dist_g <= near_v))
    si, bi = np.nonzero(kb)
    buses = pd.DataFrame({"sidx": si.astype(np.int32), "bus": bi.astype(np.int32),
                          "vm_true": A["Vm"][kb].astype(np.float32), "vm_genco": A["Vm_g"][kb].astype(np.float32),
                          "vmin": vmin[kb].astype(np.float32), "vmax": vmax[kb].astype(np.float32)})
    scen = pd.DataFrame({
        "scenario": A["scen"], "res_p99": r_p99,
        "n_lines_out": np.bincount(A["sc"][~A["on"]], minlength=S),
        "n_overload_true": np.bincount(A["sc"][rated], weights=(ld_t[rated] >= 1.0), minlength=S),
        "n_vviol_true": (PQ & ((A["Vm"] < vmin) | (A["Vm"] > vmax))).sum(axis=1),
        "err_vm_max": np.nanmax(np.where(PQ, np.abs(A["Vm_g"] - A["Vm"]), np.nan), axis=1),
        "worst_bus": np.nanargmax(np.where(PQ, np.abs(A["Vm_g"] - A["Vm"]), -1.0), axis=1),
        "err_load_max": _scen_max(np.abs(ld_g - ld_t), A["sc"], rated, S),
        "n_missed_overload_genco": np.bincount(A["sc"][rated], weights=((ld_t[rated] >= 1.0) & (ld_g[rated] < 1.0)), minlength=S),
        "n_missed_vviol_genco": (PQ & ((A["Vm"] < vmin) | (A["Vm"] > vmax)) & (A["Vm_g"] >= vmin) & (A["Vm_g"] <= vmax)).sum(axis=1),
        "dc_failed": dc_failed,
    })
    scen["worst_bus_kv"] = A["vn_kv"][np.arange(S), scen["worst_bus"].to_numpy()] if "vn_kv" in A else np.nan
    checks = {"truth_mismatch_max_MW": float(np.nanmax(np.abs(np.where(A["REF"], 0, P_t)))),
              "va_unit": A["va_unit"]}
    return lines, buses, scen, checks


# ---------------------------------------------------------------- margins
def fit_exponent(e, r, n_bins=8, q=0.99, min_count=200):
    """Slope of log(quantile of error) against log(scenario residual), from bins of the residual. Clipped to [0, 2]."""
    e, r = np.asarray(e, float), np.asarray(r, float)
    ok = np.isfinite(e) & np.isfinite(r) & (r > 0)
    e, r = e[ok], r[ok]
    if len(e) < 3 * min_count:
        return 0.0, []
    if len(e) < n_bins * min_count:
        n_bins = max(3, len(e) // min_count)
    edges = np.unique(np.quantile(r, np.linspace(0, 1, n_bins + 1)))
    idx = np.clip(np.searchsorted(edges, r, side="right") - 1, 0, len(edges) - 2)
    xs, ys = [], []
    for b in range(len(edges) - 1):
        m = idx == b
        if m.sum() >= min_count:
            qq = np.quantile(e[m], q)
            if qq > 0:
                xs.append(np.log(np.median(r[m]))); ys.append(np.log(qq))
    if len(xs) < 3:
        return 0.0, list(zip(xs, ys))
    return float(np.clip(np.polyfit(xs, ys, 1)[0], 0.0, 2.0)), list(zip(xs, ys))


def scale_of(r, r_ref, b):
    return np.power(np.maximum(np.asarray(r, float), 1e-9) / r_ref, b)


# ---------------------------------------------------------------- screening curves
def screening_curve(score, positive, sidx_global, n_scen, thresholds):
    """Flag an item if score >= threshold. Returns recall, false alarms per scenario, share of scenarios with a flag."""
    score = np.asarray(score, float); positive = np.asarray(positive, bool)
    ok = np.isfinite(score)
    score, positive, sidx = score[ok], positive[ok], np.asarray(sidx_global)[ok]
    n_pos = max(int(positive.sum()), 1)
    order = np.argsort(-score, kind="stable")
    s_desc = -score[order]                          # ascending
    cum_pos = np.r_[0, np.cumsum(positive[order])]
    cum_neg = np.r_[0, np.cumsum(~positive[order])]
    smax = np.full(n_scen, -np.inf); np.maximum.at(smax, sidx, score)
    smax_sorted = np.sort(smax)
    th = np.asarray(thresholds, float)
    n = np.searchsorted(s_desc, -th, side="right")   # items with score >= th
    tp, fp = cum_pos[n], cum_neg[n]
    sent = n_scen - np.searchsorted(smax_sorted, th, side="left")
    return pd.DataFrame({"threshold": th, "recall": tp / n_pos, "misses": n_pos - tp,
                         "false_alarms_per_scenario": fp / n_scen, "share_scenarios_flagged": sent / n_scen})


# ---------------------------------------------------------------- Newton-Raphson (polar, MATPOWER formulation)
def build_ybus(nb, f, t, Yff, Yft, Ytf, Ytt, GS, BS):
    rows = np.concatenate([f, f, t, t, np.arange(nb)])
    cols = np.concatenate([f, t, f, t, np.arange(nb)])
    vals = np.concatenate([Yff, Yft, Ytf, Ytt, GS + 1j * BS])
    return sp.csr_matrix((vals, (rows, cols)), shape=(nb, nb))


def dsbus_dv(Ybus, V):
    Ibus = Ybus @ V
    dV, dI, dVn = sp.diags(V), sp.diags(Ibus), sp.diags(V / np.abs(V))
    dS_dVm = dV @ (Ybus @ dVn).conj() + dI.conj() @ dVn
    dS_dVa = 1j * dV @ (dI - Ybus @ dV).conj()
    return dS_dVm.tocsr(), dS_dVa.tocsr()


def newton(Ybus, Sspec, V0, pv, pq, tol=1e-8, max_it=20, n_steps=None):
    """Returns final V, iterations done, list of max mismatch (p.u.) after each iteration, first-step correction size."""
    V = V0.astype(complex).copy()
    Va, Vm = np.angle(V), np.abs(V)
    pvpq = np.r_[pv, pq]
    n1 = len(pvpq)

    def mis(V):
        S = V * np.conj(Ybus @ V) - Sspec
        return np.r_[S[pvpq].real, S[pq].imag]

    F = mis(V); norms = [float(np.max(np.abs(F)))]; it = 0; first_step = np.nan
    while True:
        if n_steps is None and (norms[-1] <= tol or it >= max_it):
            break
        if n_steps is not None and it >= n_steps:
            break
        dS_dVm, dS_dVa = dsbus_dv(Ybus, V)
        J = sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                       sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc")
        try:
            dx = -splu(J).solve(F)
        except RuntimeError:
            norms.append(np.inf); break
        if it == 0:
            first_step = float(np.max(np.abs(dx)))
        Va[pvpq] += dx[:n1]; Vm[pq] += dx[n1:]
        V = Vm * np.exp(1j * Va)
        F = mis(V); it += 1; norms.append(float(np.max(np.abs(F))))
        if not np.isfinite(norms[-1]) or norms[-1] > 1e6:
            break
    return V, it, norms, first_step


def scenario_system(A, i):
    """Ybus, specified power and bus sets for scenario i of a chunk."""
    nb = A["nb"]
    m = (A["sc"] == i) & A["on"]
    Y = build_ybus(nb, A["f"][m], A["t"][m], A["Yff"][m], A["Yft"][m], A["Ytf"][m], A["Ytt"][m], A["GS"][i], A["BS"][i])
    Sspec = (A["Pg"][i] - A["Pd"][i] + 1j * (A["Qg"][i] - A["Qd"][i])) / SN_MVA
    pv, pq = np.nonzero(A["PV"][i])[0], np.nonzero(A["PQ"][i])[0]
    return Y, Sspec, pv, pq


def extract_scenario(A, i):
    """Keep only what Newton needs for scenario i (small, so it can be stored)."""
    m = A["sc"] == i
    out = {key: A[key][i].copy() for key in ["Pd", "Qd", "Pg", "Qg", "Vm", "Va", "PQ", "PV", "REF", "GS", "BS",
                                             "Va_dc", "Vm_g", "Va_g", "min_vm_pu", "max_vm_pu", "vn_kv"] if key in A}
    for key in ["f", "t", "Yff", "Yft", "Ytf", "Ytt", "rate", "on", "line"]:
        if key in A:
            out[key] = A[key][m].copy()
    out["nb"] = A["nb"]; out["scenario"] = int(A["scen"][i])
    return out


def run_newton_cases(d, tol=1e-8, max_it=20):
    """Flat start, DC start, GENCO start (to convergence) and GENCO + exactly one step, for one stored scenario."""
    nb = d["nb"]
    on = d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq, ref = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0], np.nonzero(d["REF"])[0]
    Vm_set = np.where(d["PQ"], 1.0, d["Vm"])          # setpoints at PV and slack, 1.0 elsewhere
    Va_ref = np.where(d["REF"], d["Va"], 0.0)
    V_true = d["Vm"] * np.exp(1j * np.radians(d["Va"]))

    def start(Vm, Va_deg):
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    starts = {"flat": start(Vm_set, Va_ref), "dc": start(np.ones(nb), np.nan_to_num(d["Va_dc"])),
              "genco": start(d["Vm_g"], d["Va_g"])}
    res = {"scenario": d["scenario"]}
    for name, V0 in starts.items():
        t0 = time.perf_counter()
        V, it, norms, step1 = newton(Y, Sspec, V0, pv, pq, tol=tol, max_it=max_it)
        res[f"seconds_{name}"] = time.perf_counter() - t0
        conv = norms[-1] <= tol
        res[f"iters_{name}"] = it if conv else np.nan
        res[f"converged_{name}"] = bool(conv)
        res[f"start_mismatch_{name}"] = norms[0]
        if conv:
            res[f"max_dV_vs_truth_{name}"] = float(np.max(np.abs(V - V_true)))
        if name == "genco":
            res["first_step_size"] = step1
    t0 = time.perf_counter()
    V1, _, norms1, _ = newton(Y, Sspec, starts["genco"], pv, pq, n_steps=1)
    res["seconds_genco_1step"] = time.perf_counter() - t0
    res["mismatch_after_1_step"] = norms1[-1]
    rated = on & (d["rate"] > 0)

    def loading(V):
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[rated] / d["rate"][rated]

    ld_t = loading(V_true)
    over_t = ld_t >= 1.0
    vio_t = d["PQ"] & ((d["Vm"] < d["min_vm_pu"]) | (d["Vm"] > d["max_vm_pu"]))
    for name, V in [("genco", starts["genco"]), ("genco_1step", V1)]:
        ld = loading(V); Vm = np.abs(V)
        vio = d["PQ"] & ((Vm < d["min_vm_pu"]) | (Vm > d["max_vm_pu"]))
        res[f"err_vm_max_{name}"] = float(np.max(np.abs(Vm - d["Vm"])[d["PQ"]])) if d["PQ"].any() else np.nan
        res[f"err_load_max_{name}"] = float(np.max(np.abs(ld - ld_t))) if len(ld) else np.nan
        res[f"missed_overloads_{name}"] = int((over_t & (ld < 1.0)).sum())
        res[f"missed_vviol_{name}"] = int((vio_t & ~vio).sum())
    res["true_overloads"] = int(over_t.sum()); res["true_vviol"] = int(vio_t.sum())
    return res


def newton_trace(Ybus, Sspec, V0, pv, pq, tol=1e-8, max_it=10):
    """List of (V, max mismatch p.u., seconds for this step) after 0, 1, 2, ... Newton steps."""
    V = V0.astype(complex).copy()
    Va, Vm = np.angle(V), np.abs(V)
    pvpq = np.r_[pv, pq]; n1 = len(pvpq)

    def mis(V):
        S = V * np.conj(Ybus @ V) - Sspec
        return np.r_[S[pvpq].real, S[pq].imag]

    F = mis(V)
    out = [(V.copy(), float(np.max(np.abs(F))), 0.0)]
    for _ in range(max_it):
        if out[-1][1] <= tol:
            break
        t0 = time.perf_counter()
        dS_dVm, dS_dVa = dsbus_dv(Ybus, V)
        J = sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                       sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc")
        try:
            dx = -splu(J).solve(F)
        except RuntimeError:
            break
        Va[pvpq] += dx[:n1]; Vm[pq] += dx[n1:]
        V = Vm * np.exp(1j * Va)
        F = mis(V)
        out.append((V.copy(), float(np.max(np.abs(F))), time.perf_counter() - t0))
        if not np.isfinite(out[-1][1]) or out[-1][1] > 1e6:
            break
    return out


def run_trace_cases(d, tol=1e-8, max_it=10, line_margin=0.0, v_margin=0.0):
    """Decision metrics after every Newton step from flat, DC and GENCO starts, for one stored scenario.
    Columns ending in _m use the safety margins (flag a line from 1 - line_margin, a bus within v_margin of a limit)."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
    PQ, lo, hi = d["PQ"], d["min_vm_pu"], d["max_vm_pu"]
    rated = on & (d["rate"] > 0)
    V_true = d["Vm"] * np.exp(1j * np.radians(d["Va"]))

    def loading(V):
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[rated] / d["rate"][rated]

    ld_t = loading(V_true)
    over_t = ld_t >= 1.0
    vio_t = PQ & ((d["Vm"] < lo) | (d["Vm"] > hi))

    def start(Vm, Va_deg):
        Vm = np.where(PQ, Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    starts = {"flat": start(np.ones(nb), np.zeros(nb)), "dc": start(np.ones(nb), np.nan_to_num(d["Va_dc"])),
              "genco": start(d["Vm_g"], d["Va_g"])}
    rows = []
    for name, V0 in starts.items():
        trace = newton_trace(Y, Sspec, V0, pv, pq, tol=tol, max_it=max_it)
        converged = trace[-1][1] <= tol
        for j, (V, mism, secs) in enumerate(trace):
            ld, Vm = loading(V), np.abs(V)
            vio = PQ & ((Vm < lo) | (Vm > hi))
            vio_m = PQ & ((Vm < lo + v_margin) | (Vm > hi - v_margin))
            fl_m = ld >= 1.0 - line_margin
            rows.append({"scenario": d["scenario"], "start": name, "steps": j, "mismatch": mism, "seconds_step": secs,
                         "converged_at_end": converged, "iterations_to_converge": len(trace) - 1 if converged else np.nan,
                         "err_vm_max": float(np.max(np.abs(Vm - d["Vm"])[PQ])) if PQ.any() else np.nan,
                         "err_load_max": float(np.max(np.abs(ld - ld_t))) if len(ld) else np.nan,
                         "missed_overloads": int((over_t & (ld < 1.0)).sum()), "false_overloads": int((~over_t & (ld >= 1.0)).sum()),
                         "missed_vviol": int((vio_t & ~vio).sum()), "false_vviol": int((~vio_t & vio).sum()),
                         "missed_overloads_m": int((over_t & ~fl_m).sum()), "false_overloads_m": int((~over_t & fl_m).sum()),
                         "missed_vviol_m": int((vio_t & ~vio_m).sum()), "false_vviol_m": int((~vio_t & vio_m).sum()),
                         "true_overloads": int(over_t.sum()), "true_vviol": int(vio_t.sum())})
    return rows


# ===================== GENCO helpers =====================
from scipy.sparse.csgraph import dijkstra


class ScenarioEval:
    """Truth-based checks of any voltage state for one stored scenario (from extract_scenario)."""

    def __init__(self, d):
        self.d = d
        nb, on = d["nb"], d["on"]
        self.Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
        self.Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
        self.pv, self.pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
        self.nonref = np.nonzero(~d["REF"])[0]
        self.rated = on & (d["rate"] > 0)
        Vt = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
        self.ld_t = self.loading(Vt)
        self.over_t = self.ld_t >= 1.0
        self.vio_t = d["PQ"] & ((d["Vm"] < d["min_vm_pu"]) | (d["Vm"] > d["max_vm_pu"]))

    def loading(self, V):
        d = self.d
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[self.rated] / d["rate"][self.rated]

    def state(self, Vm, Va_deg):
        d = self.d
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    def metrics(self, Vm, Va_deg):
        d = self.d
        V = self.state(Vm, Va_deg)
        S = V * np.conj(self.Y @ V) - self.Sspec
        Pm, Qm = np.abs(S.real), np.abs(S.imag)
        ld = self.loading(V)
        Vabs = np.abs(V)
        vio = d["PQ"] & ((Vabs < d["min_vm_pu"]) | (Vabs > d["max_vm_pu"]))
        err = np.abs(Vabs - d["Vm"])
        return {"err_vm_max": float(err[d["PQ"]].max()) if d["PQ"].any() else np.nan,
                "err_load_max": float(np.max(np.abs(ld - self.ld_t))) if len(ld) else np.nan,
                "mismatch_pu": float(max(Pm[np.r_[self.pv, self.pq]].max(), Qm[self.pq].max())),
                "res_p99_MW": float(np.percentile(Pm[self.nonref], 99) * SN_MVA),
                "missed_overloads": int((self.over_t & (ld < 1.0)).sum()),
                "missed_vviol": int((self.vio_t & ~vio).sum())}

    def bus_errors(self, Vm):
        return np.where(self.d["PQ"], np.abs(Vm - self.d["Vm"]), np.nan)


def thevenin_pq(d, block=200):
    """|Z_ii| for every PQ bus with PV and slack buses held at fixed voltage: how much the voltage at a bus
    moves per unit of current injected there (large = electrically weak spot)."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    pq = np.nonzero(d["PQ"])[0]
    lu = splu(Y[pq][:, pq].tocsc())
    n = len(pq); z = np.empty(n, complex)
    for s in range(0, n, block):
        idx = np.arange(s, min(n, s + block))
        E = np.zeros((n, len(idx)), complex); E[idx, np.arange(len(idx))] = 1.0
        X = lu.solve(E)
        z[idx] = X[idx, np.arange(len(idx))]
    return pq, np.abs(z)


def hops_to_voltage_control(d):
    """Number of in-service lines between each bus and the nearest PV or slack bus."""
    nb, on = d["nb"], d["on"]
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    src = np.nonzero(d["PV"] | d["REF"])[0]
    dist = dijkstra(A, directed=False, unweighted=True, indices=src, min_only=True)
    return dist


def dc_solve_seconds(d, reps=3):
    """Time of one DC power flow (one sparse solve on the susceptance matrix), for the cost comparison."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    keep = np.nonzero(~d["REF"])[0]
    P = ((d["Pg"] - d["Pd"]) / SN_MVA)[keep]
    t0 = time.perf_counter()
    for _ in range(reps):
        B = (-Y.imag)[keep][:, keep].tocsc()
        splu(B).solve(P)
    return (time.perf_counter() - t0) / reps


# ===================== outage construction =====================
from scipy.sparse.csgraph import connected_components
try:
    from lightsim2grid.algorithm import NR_KLU as _KLU
    HAVE_KLU = True
except Exception:
    try:
        from lightsim2grid.solver import KLUSolver as _KLU
        HAVE_KLU = True
    except Exception:
        HAVE_KLU = False


def system_of(d):
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    S = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    return Y, S, np.nonzero(d["REF"])[0], np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]


def klu_solve(Y, S, V0, ref, pv, pq, max_iter, tol=1e-8):
    # lightsim2grid's C++ Newton-Raphson with the KLU sparse solver. Returns V, converged, iterations, seconds.
    w = np.zeros(Y.shape[0]); w[ref] = 1.0 / len(ref)
    Yc = Y.tocsc()
    s = _KLU()
    t0 = time.perf_counter()
    ok = s.solve(Yc, np.asarray(V0, complex), np.asarray(S, complex), ref.astype(np.int32), w,
                 pv.astype(np.int32), pq.astype(np.int32), int(max_iter), float(tol))
    dt = time.perf_counter() - t0
    V = np.array(s.get_V(), dtype=complex, copy=True)
    return V, bool(ok), int(s.get_nb_iter()), dt


def dc_angles(d):
    # DC power flow on the current topology (series susceptances, no shunts, no phase shifts). Returns angles in degrees.
    nb, on = d["nb"], d["on"]
    f, t, b = d["f"][on], d["t"][on], d["Yft"][on].imag
    B = sp.csr_matrix((np.r_[b, b, -b, -b], (np.r_[f, t, f, t], np.r_[f, t, t, f])), shape=(nb, nb)).tocsc()
    ref = np.nonzero(d["REF"])[0]
    keep = np.setdiff1d(np.arange(nb), ref)
    P = ((d["Pg"] - d["Pd"]) / SN_MVA)[keep]
    th = np.zeros(nb)
    th[keep] = splu(B[keep][:, keep]).solve(P)
    return np.degrees(th) + float(d["Va"][ref[0]])


def connected(nb, f, t):
    A = sp.csr_matrix((np.ones(len(f)), (f, t)), shape=(nb, nb))
    return connected_components(A, directed=False)[0] == 1


def pick_outages(d, extra, mode, rng):
    # Indices (into the scenario's branch rows) of `extra` more lines to remove, keeping the grid in one piece.
    # mode "clustered": the lines nearest (in hops) to a random centre bus, like a storm footprint.
    # mode "uniform": random lines anywhere, like the released test sets.
    nb = d["nb"]; on = d["on"].copy()
    cand = np.nonzero(on & (d["rate"] > 0))[0]
    hop = None
    if mode == "clustered":
        center = int(rng.integers(nb))
        A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
        hop = dijkstra(A, directed=False, unweighted=True, indices=center)
        dist = np.minimum(hop[d["f"][cand]], hop[d["t"][cand]])
        order = cand[np.lexsort((rng.random(len(cand)), dist))]
    else:
        order = rng.permutation(cand)
    removed = []
    for i in order:
        if len(removed) == extra:
            break
        trial = on.copy(); trial[i] = False
        if connected(nb, d["f"][trial], d["t"][trial]):
            on = trial; removed.append(int(i))
    if len(removed) < extra:
        return None, None
    radius = float(np.max(np.minimum(hop[d["f"][removed]], hop[d["t"][removed]]))) if hop is not None else np.nan
    return np.array(removed), radius


def solve_new_topology(d, removed, tol=1e-8, max_iter=30):
    # Exact AC solution after removing `removed`; starts: the base solution, then flat, then DC.
    d2 = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in d.items()}
    d2["on"][removed] = False
    Y, S, ref, pv, pq = system_of(d2)
    base = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
    flat = np.where(d2["PQ"], 1.0, d2["Vm"]) * np.exp(1j * np.radians(np.where(d2["REF"], d2["Va"], 0.0)))
    d2["Va_dc"] = dc_angles(d2)
    dc = np.where(d2["PQ"], 1.0, d2["Vm"]) * np.exp(1j * np.radians(d2["Va_dc"]))
    for name, V0 in [("base", base), ("flat", flat), ("dc", dc)]:
        if HAVE_KLU:
            V, ok, it, _ = klu_solve(Y, S, V0, ref, pv, pq, max_iter, tol)
        else:
            V, it, norms, _ = newton(Y, S, V0, pv, pq, tol=tol, max_it=max_iter)
            ok = norms[-1] <= tol
        if ok:
            d2["Vm"], d2["Va"] = np.abs(V), np.degrees(np.angle(V))
            d2["solved_from"] = name
            return d2
    return None




"""J1 core: true AC power-flow voltage sensitivities from the Jacobian, a finite-difference check of them,
hop distances in the grid graph, and the comparison metrics. Uses build_ybus / dsbus_dv / newton / SN_MVA from
the step-1/6 core."""
import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import shortest_path
from scipy.sparse.linalg import splu


def ac_system(d):
    """Ybus, true complex voltages, specified injections (p.u.) and bus sets of one stored scenario."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    V = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
    S = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
    return Y, V, S, pv, pq


def power_flow_jacobian(Y, V, pv, pq):
    """Jacobian of [P mismatch at pv+pq; Q mismatch at pq] with respect to [angle at pv+pq; magnitude at pq]."""
    pvpq = np.r_[pv, pq]
    dS_dVm, dS_dVa = dsbus_dv(Y, V)
    return sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                      sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc"), pvpq


def true_sensitivity_rows(d, targets):
    """For each target bus (PQ in this scenario): dVm_target / dPd_j and dVm_target / dQd_j for every bus j,
    in p.u. voltage per p.u. load (100 MW or 100 Mvar). A load increase is a lower injection, hence the minus sign.
    Loads at the slack bus (P) and at voltage-controlled buses (Q) do not move any voltage: those entries are 0."""
    Y, V, S, pv, pq = ac_system(d)
    J, pvpq = power_flow_jacobian(Y, V, pv, pq)
    lu = splu(J.T.tocsc())
    n1, nb = len(pvpq), d["nb"]
    pos_pq = {int(b): n1 + i for i, b in enumerate(pq)}
    out = {}
    for t in targets:
        if int(t) not in pos_pq:
            continue
        e = np.zeros(J.shape[0]); e[pos_pq[int(t)]] = 1.0
        y = lu.solve(e)
        TP, TQ = np.zeros(nb), np.zeros(nb)
        TP[pvpq] = -y[:n1]
        TQ[pq] = -y[n1:]
        out[int(t)] = (TP, TQ)
    return out


def ac_fd_check(d, t, delta_pu=0.01):
    """Finite-difference check of the Jacobian sensitivities. Converge the base case, raise the reactive load at
    bus t by delta_pu (1 Mvar for 0.01), converge again, and compare dVm_t / dQd_t from the two solutions with the
    Jacobian value at the base solution. Returns (finite difference, Jacobian)."""
    Y, V, S, pv, pq = ac_system(d)
    V0, _, n0, _ = newton(Y, S, V, pv, pq, tol=1e-11, max_it=20)
    S2 = S.copy(); S2[t] -= 1j * delta_pu
    V2, _, n2, _ = newton(Y, S2, V0, pv, pq, tol=1e-11, max_it=20)
    if not n0 or not n2 or n0[-1] > 1e-9 or n2[-1] > 1e-9:
        return np.nan, np.nan
    J, pvpq = power_flow_jacobian(Y, V0, pv, pq)
    row = np.nonzero(pq == t)[0]
    e = np.zeros(J.shape[0]); e[len(pvpq) + int(row[0])] = 1.0
    jac = float(-splu(J.T.tocsc()).solve(e)[len(pvpq) + int(row[0])])
    return float((np.abs(V2[t]) - np.abs(V0[t])) / delta_pu), jac


def hop_distances(d, targets):
    """Number of in-service lines between each target and every bus (inf if not connected)."""
    nb, on = d["nb"], d["on"]
    f, t = d["f"][on], d["t"][on]
    A = sp.csr_matrix((np.ones(2 * len(f)), (np.r_[f, t], np.r_[t, f])), shape=(nb, nb))
    return shortest_path(A, unweighted=True, directed=False, indices=np.asarray(targets, dtype=np.int64))


def row_metrics(gP, gQ, tP, tQ, pq_mask, pv_mask):
    """Compare GENCO's sensitivity row with the true one (same units). Only loads that can move voltages enter the
    gain and cosine: P at every non-slack bus, Q at PQ buses. Leakage: share of GENCO's reactive-load sensitivity that
    sits on voltage-controlled buses, where the true value is exactly 0."""
    g = np.r_[gP[pq_mask | pv_mask], gQ[pq_mask]]
    tr = np.r_[tP[pq_mask | pv_mask], tQ[pq_mask]]
    tt = float(tr @ tr)
    gain = float(g @ tr / tt) if tt > 0 else np.nan
    cos = float(g @ tr / (np.linalg.norm(g) * np.sqrt(tt))) if tt > 0 and np.linalg.norm(g) > 0 else np.nan
    rel_err = float(np.linalg.norm(g - tr) / np.sqrt(tt)) if tt > 0 else np.nan
    leak = float(np.abs(gQ[~pq_mask]).sum() / max(np.abs(gQ).sum(), 1e-30))
    gq, tq = gQ[pq_mask], tQ[pq_mask]
    tqq = float(tq @ tq)
    gain_q = float(gq @ tq / tqq) if tqq > 0 else np.nan
    cos_q = float(gq @ tq / (np.linalg.norm(gq) * np.sqrt(tqq))) if tqq > 0 and np.linalg.norm(gq) > 0 else np.nan
    return {"gain": gain, "cosine": cos, "relative_error": rel_err, "gain_Q": gain_q, "cosine_Q": cos_q, "pv_q_leakage": leak}


def hop_profile(abs_row, hops, max_hop=8):
    """Sum of |sensitivity| at each hop distance 0..max_hop (the last bin holds max_hop and beyond)."""
    h = np.where(np.isfinite(hops), np.minimum(hops, max_hop), max_hop).astype(int)
    return np.bincount(h, weights=abs_row, minlength=max_hop + 1)[: max_hop + 1]


def slope_through(x, y):
    """Least-squares slope of y on x (both centred): how strongly GENCO's voltage follows the true one."""
    x = np.asarray(x, float) - np.mean(x); y = np.asarray(y, float) - np.mean(y)
    return float(x @ y / (x @ x)) if x @ x > 0 else np.nan


"""J2 core: true AC responses to realistic changes (one reactive load, all loads together, one generator voltage
set-point), from the Jacobian at the true solution, plus AC re-solves to check that the linear answer holds at
these sizes. Uses ac_system / power_flow_jacobian from J1 and newton / dsbus_dv / SN_MVA from the step-1 core."""
import numpy as np
from scipy.sparse.linalg import splu


class TrueResponses:
    """Linear AC responses of all bus voltage magnitudes at the true solution of one scenario."""

    def __init__(self, d):
        self.d = d
        self.Y, self.V, self.S, self.pv, self.pq = ac_system(d)
        self.J, self.pvpq = power_flow_jacobian(self.Y, self.V, self.pv, self.pq)
        self.lu = splu(self.J.tocsc())
        self.n1, self.nb = len(self.pvpq), d["nb"]

    def _dvm(self, rhs):
        dx = -self.lu.solve(rhs)
        out = np.zeros(self.nb)
        out[self.pq] = dx[self.n1:]
        return out

    def reactive_load(self, j):
        """dVm (all buses) per p.u. (100 Mvar) more reactive load at bus j; zero if j holds its voltage."""
        pos = np.nonzero(self.pq == j)[0]
        if len(pos) == 0:
            return np.zeros(self.nb)
        rhs = np.zeros(self.J.shape[0]); rhs[self.n1 + int(pos[0])] = 1.0
        return self._dvm(rhs)

    def load_growth(self):
        """dVm (all buses) per unit of growth of every load (P and Q, constant power factor); the slack takes the extra
        active power, generator set-points stay as they are."""
        Pd, Qd = self.d["Pd"] / SN_MVA, self.d["Qd"] / SN_MVA
        rhs = np.r_[Pd[self.pvpq], Qd[self.pq]]
        return self._dvm(rhs)

    def setpoint(self, g):
        """dVm (all buses) per p.u. higher voltage set-point at the voltage-controlled bus g (itself: 1)."""
        dS_dVm, _ = dsbus_dv(self.Y, self.V)
        col = dS_dVm[:, g].toarray().ravel()
        rhs = np.r_[col[self.pvpq].real, col[self.pq].imag]
        out = self._dvm(rhs)
        out[g] = 1.0
        return out

    def resolve(self, dQ=None, growth=0.0, dV=None):
        """AC power flow solved again after a change; returns the new |V| (nan if no convergence)."""
        S = self.S.copy()
        V0 = self.V.copy()
        if dQ is not None:
            j, q = dQ
            S[j] -= 1j * q
        if growth:
            S -= growth * (self.d["Pd"] + 1j * self.d["Qd"]) / SN_MVA
        if dV is not None:
            g, v = dV
            V0[g] = (abs(V0[g]) + v) * np.exp(1j * np.angle(V0[g]))
        V2, _, norms, _ = newton(self.Y, S, V0, self.pv, self.pq, tol=1e-10, max_it=25)
        return np.abs(V2) if norms and norms[-1] < 1e-8 else np.full(self.nb, np.nan)


def nearest(hops_row, mask, n, lo=0, hi=np.inf):
    """Up to n buses in `mask` whose hop distance lies in [lo, hi], nearest first."""
    idx = np.nonzero(mask & (hops_row >= lo) & (hops_row <= hi) & np.isfinite(hops_row))[0]
    return idx[np.argsort(hops_row[idx], kind="stable")][:n]


def largest(values, mask, n):
    """Up to n buses in `mask` with the largest |values|."""
    idx = np.nonzero(mask)[0]
    return idx[np.argsort(-np.abs(values[idx]), kind="stable")][:n]


import json, zipfile, warnings, platform
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor
from gridfm_graphkit.datasets.globals import PD_H, QD_H, VM_H

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(SEED)
RAW3_PAT = re.compile(r"(?:^|/)(bus_data|branch_data|gen_data)\.parquet/scenario_partition=(\d+)/[^/]+$")


def read_all(path, scen):
    dset = ds.dataset(str(path), format="parquet", partitioning="hive")
    values = pa.array(np.asarray(scen, dtype=np.int64)).cast(dset.schema.field("scenario").type)
    t = dset.to_table(filter=ds.field("scenario").isin(values)).to_pandas()
    return t.drop(columns=[c for c in ["scenario_partition"] if c in t.columns])


def scenario_dicts(level, ids, vm_g, va_g):
    """Stored scenarios with the true state (from the data) and GENCO's state (vm_g, va_g: scenario -> array, rad)."""
    ids = np.sort(np.asarray(ids, dtype=np.int64))
    pred = pd.concat([pd.DataFrame({"scenario": int(s), "bus": np.arange(len(vm_g[int(s)])), "vm": vm_g[int(s)],
                                    "va": va_g[int(s)]}) for s in ids], ignore_index=True)
    A = load_chunk(find_local(LOCAL_DIR, level, "bus_data"), find_local(LOCAL_DIR, level, "branch_data"), pred, ids)
    return {int(ids[i]): extract_scenario(A, i) for i in range(len(ids))}


print("ready")

## 1. Download: model, reproduction check, and two data blocks per outage level

In [ ]:
from huggingface_hub import HfApi, snapshot_download
print("Listing the repository ...")
all_files = [(f.path, f.size) for f in HfApi().list_repo_tree(REPO_ID, recursive=True, revision=HF_REVISION) if getattr(f, "size", None) is not None]
local_has = lambda p: (Path(LOCAL_DIR) / p).exists()
plan = {p: s for p, s in all_files if p.startswith(MODEL_DIR + "/")}
plan.update({p: s for p, s in all_files if p.startswith(f"mlflow/eval/test_{COMPARE_LEVEL}/")
             and (p.endswith("predictions.parquet") or p.endswith("meta.yaml")) and not re.search(r"_(gen|branch)_predictions", p)})
for k in LEVELS:
    for p, s in all_files:
        m = RAW3_PAT.search(p) if p.startswith(f"data/test_{k}/") else None
        if m and int(m.group(2)) in PARTS:
            plan[p] = s
new = sum(s for p, s in plan.items() if not local_has(p)) / 1e9
print(f"Planned: {len(plan)} files, {new:.2f} GB not yet on disk")
if new > MAX_DOWNLOAD_GB:
    raise RuntimeError("Planned download is above MAX_DOWNLOAD_GB")
snapshot_download(REPO_ID, allow_patterns=list(plan), local_dir=LOCAL_DIR, revision=HF_REVISION)
print("Download finished")

## 2. Scenarios, graphs and the model

In [ ]:
SEL = {}
for k in LEVELS:
    on_disk = scenario_ids(find_local(LOCAL_DIR, k, "bus_data"))
    SEL[k] = np.sort(RNG.choice(on_disk, min(N_PER_LEVEL, len(on_disk)), replace=False)).astype(np.int64)
    print(f"N-{k}: {len(SEL[k])} scenarios")

In [ ]:
# [GPU] build graphs

import yaml
cfg_path = os.path.join(GRAPHKIT_DIR, "scripts", "contingency", "configs", "caseTexas_eval.yaml")
with open(cfg_path) as f:
    ARGS = NestedNamespace(**yaml.safe_load(f))
stats = torch.load(os.path.join(LOCAL_DIR, MODEL_DIR, "normalizer_stats.pt"), map_location="cpu", weights_only=True)
stats_net = stats[NETWORK] if NETWORK in stats else (stats if "baseMVA" in stats else next(iter(stats.values())))
NORMALIZER = load_normalizer(ARGS)
NORMALIZER.fit_from_dict(stats_net)
TRANSFORM = get_task_transforms(ARGS)
print("normalizer baseMVA:", NORMALIZER.baseMVA, "| config:", cfg_path)

GRAPHS = {}
for k in LEVELS:
    bus = read_all(find_local(LOCAL_DIR, k, "bus_data"), SEL[k])
    gen = read_all(find_local(LOCAL_DIR, k, "gen_data"), SEL[k])
    br = read_all(find_local(LOCAL_DIR, k, "branch_data"), SEL[k])
    bus = merge_agg_gen_into_bus(bus, gen)
    bg, gg, rg = bus.groupby("scenario"), gen.groupby("scenario"), br.groupby("scenario")
    GRAPHS[k] = []
    for s in SEL[k]:
        b = bg.get_group(s).sort_values("bus").reset_index(drop=True)
        data = build_hetero_data_for_scenario(int(s), b, gg.get_group(s), rg.get_group(s))
        NORMALIZER.transform(data)
        GRAPHS[k].append(TRANSFORM(data))
    print(f"k={k}: {len(GRAPHS[k])} graphs built")
    del bus, gen, br

In [ ]:
# [GPU] load model and smoke test

TASK = get_task(ARGS, [NORMALIZER])
sd = torch.load(os.path.join(LOCAL_DIR, MODEL_DIR, "best_model_state_dict.pt"), map_location="cpu")
sd = adapt_state_dict_for_model(sd, TASK)
missing_keys, unexpected_keys = TASK.load_state_dict(sd, strict=False)
print(f"checkpoint loaded: {len(sd)} tensors | missing {len(missing_keys)} | unexpected {len(unexpected_keys)}")
if missing_keys:
    raise RuntimeError(f"The checkpoint does not match the model. Missing: {missing_keys[:5]}")
MODEL = TASK.model.to(DEVICE).eval()
print("GENCO parameters:", sum(p.numel() for p in MODEL.parameters()))


def forward(model, batch):
    return model(batch.x_dict, batch.edge_index_dict, batch.edge_attr_dict, batch.mask_dict)


test_ids = SEL[COMPARE_LEVEL][:4]
rel = read_cols(find_pred(LOCAL_DIR, COMPARE_LEVEL), ["scenario", "bus", "vm_pu", "va"], scen=test_ids).sort_values(["scenario", "bus"])
with torch.no_grad():
    b = next(iter(DataLoader(GRAPHS[COMPARE_LEVEL][:4], batch_size=4))).to(DEVICE)
    out = forward(MODEL, b)["bus"].float().cpu().numpy()
d_vm = np.abs(out[:, VM_OUT] - rel["vm_pu"].to_numpy())
d_va = np.abs(out[:, VA_OUT] - rel["va"].to_numpy())
print(f"Smoke test vs released predictions (4 scenarios): Vm max diff {d_vm.max():.2e} p.u., Va max diff {d_va.max():.2e} rad")
if np.median(d_vm) > 1e-3:
    raise RuntimeError("Our run does not reproduce the released predictions; check the installation and the downloaded files.")
print("OK: our GENCO run reproduces the released predictions.")

## 3. Run GENCO on every scenario and find where it fails (as in the Jacobian probe)

In [ ]:
VMG, VAG, OUT4, VMT, PQM, PVM = {}, {}, {}, {}, {}, {}
POS = {}
t0 = time.time()
for k in LEVELS:
    truth = read_cols(find_local(LOCAL_DIR, k, "bus_data"), ["scenario", "bus", "Vm", "PQ", "PV"], scen=SEL[k])
    truth["scenario"] = truth["scenario"].astype(np.int64)
    truth = truth.sort_values(["scenario", "bus"], kind="stable")
    TG = {int(s): g for s, g in truth.groupby("scenario")}
    for i, s in enumerate(SEL[k]):
        POS[(k, int(s))] = i
    for b in DataLoader(GRAPHS[k], batch_size=BATCH, shuffle=False):
        b = b.to(DEVICE)
        with torch.no_grad():
            out = forward(MODEL, b)["bus"].float().cpu().numpy()
        ptr = b["bus"].ptr.cpu().tolist()
        for g_, s in enumerate(b["scenario_id"].view(-1).cpu().tolist()):
            key = (k, int(s)); o = out[ptr[g_]:ptr[g_ + 1]]
            VMG[key], VAG[key], OUT4[key] = o[:, VM_OUT].astype(np.float64), o[:, VA_OUT].astype(np.float64), o
            VMT[key] = TG[int(s)]["Vm"].to_numpy(np.float64); PQM[key] = TG[int(s)]["PQ"].to_numpy() > 0.5
            PVM[key] = TG[int(s)]["PV"].to_numpy() > 0.5
KEYS = list(VMG)
NB = len(VMT[KEYS[0]])
ERR = np.stack([np.where(PQM[key], VMG[key] - VMT[key], np.nan) for key in KEYS])
WORST = np.nanmax(np.abs(ERR), axis=1)
HARD = {key: bool(w > HARD_ERR) for key, w in zip(KEYS, WORST)}
print(f"GENCO run on {len(KEYS)} scenarios in {time.time() - t0:.0f} s; {NB} buses")
display(pd.DataFrame({"k": [key[0] for key in KEYS], "hard": [HARD[key] for key in KEYS]}).groupby("k").hard.agg(["size", "sum", "mean"])
        .rename(columns={"size": "scenarios", "sum": "hard (worst error > HARD_ERR)", "mean": "share"}))

# target buses: bus 1797, the buses GENCO most often gets wrong, and random PQ buses
pq_share = np.mean(np.stack([PQM[key] for key in KEYS]), axis=0)
cand = np.nonzero(pq_share >= 0.95)[0]
miss_count = np.nansum(np.abs(ERR) > HARD_ERR, axis=0)
watch = [WATCH_BUS] if WATCH_BUS in set(cand.tolist()) else []
often = [int(b) for b in np.argsort(-miss_count, kind="stable") if b in set(cand.tolist()) and b not in watch and miss_count[b] > 0]
often = often[:N_TARGET_MISSED]
rest = np.setdiff1d(cand, np.array(watch + often, dtype=int))
rand = sorted(np.random.default_rng(SEED + 2).choice(rest, min(N_TARGET_RANDOM, len(rest)), replace=False).tolist())
TARGETS = np.array(watch + often + rand, dtype=np.int64)
GROUP = {**{b: "bus 1797" for b in watch}, **{b: "often missed" for b in often}, **{b: "random" for b in rand}}
print(f"{len(TARGETS)} target buses: {len(watch)} watch bus, {len(often)} often missed, {len(rand)} random")
display(pd.DataFrame({"bus": TARGETS, "group": [GROUP[int(b)] for b in TARGETS],
                      "scenarios with error > HARD_ERR": miss_count[TARGETS].astype(int)}).head(12))

# scenarios whose sensitivities are computed: hard and normal ones at each level
GRAD_KEYS = []
for k in LEVELS:
    hk = [key for key in KEYS if key[0] == k and HARD[key]]
    nk = [key for key in KEYS if key[0] == k and not HARD[key]]
    pick = lambda L, n: [L[i] for i in sorted(RNG.choice(len(L), min(n, len(L)), replace=False))] if L else []
    GRAD_KEYS += pick(hk, N_GRAD_HARD) + pick(nk, N_GRAD_NORMAL)
print(f"sensitivities will be computed for {len(GRAD_KEYS)} scenarios "
      f"({sum(HARD[key] for key in GRAD_KEYS)} hard, {sum(not HARD[key] for key in GRAD_KEYS)} normal)")

## 4. Targets and scenarios (the same as in the Jacobian probe)

In [ ]:
J1_TARGETS = [1797, 1869, 1872, 1873, 1874, 1875, 1870, 499, 548, 1557, 153, 117, 185, 222, 388, 412, 540, 572, 613, 678,
              811, 881, 1089, 1176, 1286, 1423, 1613, 1621, 1651, 1751, 1980]
WRONG = [1797, 1869, 1870, 1872, 1873, 1874, 1875, 1557]      # Jacobian probe: GENCO's sensitivity pattern is wrong (cosine 0.1-0.3)
RIGHT_MISSED = [153, 499, 548]                                 # Jacobian probe: often missed, but the pattern is right
if NB > max(J1_TARGETS):
    TARGETS = np.array(J1_TARGETS, dtype=np.int64)
    GROUP = {b: ("wrong pattern" if b in WRONG else "often missed, right pattern" if b in RIGHT_MISSED else "random") for b in J1_TARGETS}
    A_TARGETS = np.array([1797, 1869, 1870, 1874, 1557, 153, 499, 117, 412, 881], dtype=np.int64)
else:   # small test grid: use the targets chosen above
    GROUP = {b: ("wrong pattern" if g != "random" else "random") for b, g in GROUP.items()}
    A_TARGETS = TARGETS[: min(6, len(TARGETS))]
J2_KEYS = []
for k in LEVELS:
    kk = [key for key in GRAD_KEYS if key[0] == k]
    J2_KEYS += [key for key in kk if HARD[key]][:J2_HARD] + [key for key in kk if not HARD[key]][:J2_NORMAL]
print(f"{len(TARGETS)} targets ({sum(g == 'wrong pattern' for g in GROUP.values())} with the wrong pattern in the Jacobian probe); "
      f"{len(A_TARGETS)} for the load-by-load test; {len(J2_KEYS)} scenarios for tests (a) and (c), {len(GRAD_KEYS)} for test (b)")


def genco_vm(graphs):
    """GENCO's voltage magnitudes for a list of graphs."""
    out = []
    for start in range(0, len(graphs), BATCH_PERT):
        part = graphs[start:start + BATCH_PERT]
        b = next(iter(DataLoader(part, batch_size=len(part), shuffle=False))).to(DEVICE)
        with torch.no_grad():
            vm = forward(MODEL, b)["bus"][:, VM_OUT].float().cpu().numpy()
        ptr = b["bus"].ptr.cpu().tolist()
        out += [vm[ptr[i]:ptr[i + 1]].astype(np.float64) for i in range(len(part))]
    return out


def perturbed(key, edit):
    g = GRAPHS[key[0]][POS[key]].clone()
    g["bus"].x = g["bus"].x.clone()
    edit(g["bus"].x)
    return g


def genco_autograd_rows(key, targets):
    """dVm_target / dQd_j (per p.u.) for the given targets, by automatic differentiation."""
    b = next(iter(DataLoader([GRAPHS[key[0]][POS[key]]], batch_size=1))).to(DEVICE)
    x = b["bus"].x.detach().clone().requires_grad_(True)
    b["bus"].x = x
    vm = forward(MODEL, b)["bus"][:, VM_OUT]
    rows = []
    for i, t in enumerate(targets):
        grad, = torch.autograd.grad(vm[int(t)], x, retain_graph=i < len(targets) - 1)
        rows.append(grad[:, QD_H].detach().float().cpu().numpy() * SN_MVA / float(NORMALIZER.baseMVA))
    return np.stack(rows)


DICTS = {}
for k in LEVELS:
    ids = [key[1] for key in GRAD_KEYS if key[0] == k]
    if ids:
        D = scenario_dicts(k, ids, {s: VMG[(k, s)] for s in ids}, {s: VAG[(k, s)] for s in ids})
        DICTS.update({(k, s): D[s] for s in ids})
print("scenario data ready")

## 5. (a) One reactive load at a time

In [ ]:
rows_a, ac_checks = [], []
t0 = time.time()
for key in J2_KEYS:
    d = DICTS[key]; T = TrueResponses(d)
    pq, pv = PQM[key], PVM[key]
    tg = [int(t) for t in A_TARGETS if pq[int(t)]]
    if not tg:
        continue
    G = genco_autograd_rows(key, tg)
    H = hop_distances(d, tg)
    picks = []
    for i, t in enumerate(tg):
        true_row = true_sensitivity_rows(d, [t])[t][1]
        picks += [(t, t, "own load")]
        picks += [(t, int(j), "near (1-2 lines)") for j in largest(true_row, pq & (H[i] >= 1) & (H[i] <= 2), N_NEAR)]
        picks += [(t, int(j), "mid (3-5 lines), GENCO's largest") for j in largest(G[i], pq & (H[i] >= 3) & (H[i] <= 5), N_MID)]
        picks += [(t, int(j), "mid (3-5 lines), truth's largest") for j in largest(true_row, pq & (H[i] >= 3) & (H[i] <= 5), N_MID)]
        picks += [(t, int(j), "far (6+ lines), GENCO's largest") for j in largest(G[i], pq & (H[i] >= 6), N_FAR)]
        picks += [(t, int(j), "generator bus (true effect 0)") for j in largest(G[i], pv & (H[i] <= 3), N_GENQ)]
    loads = sorted({j for _, j, _ in picks})
    dq = DQ_MVAR / float(NORMALIZER.baseMVA)
    graphs = []
    for j in loads:
        for sgn in (1, -1):
            graphs.append(perturbed(key, lambda x, j=j, sgn=sgn: x.__setitem__((j, QD_H), x[j, QD_H] + sgn * dq)))
    vms = genco_vm(graphs)
    sec = {j: (vms[2 * n] - vms[2 * n + 1]) / (2 * DQ_MVAR / SN_MVA) for n, j in enumerate(loads)}
    col = {j: T.reactive_load(j) for j in loads}
    for t, j, cls in picks:
        i = tg.index(t)
        rows_a.append({"k": key[0], "scenario": key[1], "hard": HARD[key], "target": t, "group": GROUP[t], "load": j, "class": cls,
                       "hops": float(H[i][j]), "autograd": float(G[i][j]), "genco_secant": float(sec[j][t]),
                       "true": float(col[j][t]), "true_own": float(col[t][t]) if t in col else np.nan})
    if len(ac_checks) < 6:
        j = next(jj for jj in loads if pq[jj])
        v_ac, v0 = T.resolve(dQ=(j, DQ_MVAR / SN_MVA)), T.resolve()
        ac_checks.append({"scenario": key[1], "load": j, "target": tg[0], "linear (Jacobian)": float(col[j][tg[0]]),
                          "AC re-solved": float((v_ac[tg[0]] - v0[tg[0]]) / (DQ_MVAR / SN_MVA))})
TA = pd.DataFrame(rows_a)
TA["secant / true"] = TA.genco_secant.abs() / TA.true.abs().where(TA.true.abs() > 0)
TA["secant / autograd"] = TA.genco_secant / TA.autograd.where(TA.autograd.abs() > 0)
TA["secant / true own"] = TA.genco_secant.abs() / TA.true_own.abs()
TA["same sign"] = np.sign(TA.genco_secant) == np.sign(TA.true)
TA.to_csv(Path(OUT_DIR) / "single_loads.csv", index=False)
pd.DataFrame(ac_checks).to_csv(Path(OUT_DIR) / "ac_check.csv", index=False)
print(f"(a) {len(TA)} target-load pairs in {time.time() - t0:.0f} s. AC check of the linear truth at {DQ_MVAR} Mvar:")
display(pd.DataFrame(ac_checks))
SUM_A = TA.groupby(["group", "class"]).agg(pairs=("load", "size"), secant_over_true=("secant / true", "median"),
                                          secant_over_autograd=("secant / autograd", "median"),
                                          secant_over_true_own=("secant / true own", "median"), same_sign=("same sign", "mean"))
print(f"GENCO's response to {DQ_MVAR} Mvar more load (secant) against the truth; medians:")
display(SUM_A)

## 6. (b) All loads grow together

In [ ]:
rows_b, glob_b = [], []
t0 = time.time()
graphs = []
for key in GRAD_KEYS:
    for sgn in (1, -1):
        graphs.append(perturbed(key, lambda x, sgn=sgn: (x[:, PD_H].mul_(1 + sgn * GROWTH), x[:, QD_H].mul_(1 + sgn * GROWTH))))
vms = genco_vm(graphs)
for n, key in enumerate(GRAD_KEYS):
    d = DICTS[key]; T = TrueResponses(d)
    g_sec = (vms[2 * n] - vms[2 * n + 1]) / (2 * GROWTH)
    vp, vmn = T.resolve(growth=GROWTH), T.resolve(growth=-GROWTH)
    t_sec = (vp - vmn) / (2 * GROWTH)
    t_lin = T.load_growth()
    pq = PQM[key]
    for t in TARGETS:
        t = int(t)
        if pq[t]:
            rows_b.append({"k": key[0], "scenario": key[1], "hard": HARD[key], "target": t, "group": GROUP[t],
                           "genco": float(g_sec[t]), "true": float(t_sec[t]), "true_linear": float(t_lin[t])})
    ok = pq & np.isfinite(t_sec)
    glob_b.append({"k": key[0], "scenario": key[1], "hard": HARD[key], "slope_all_pq": slope_through(t_sec[ok], g_sec[ok]),
                   "corr_all_pq": float(np.corrcoef(t_sec[ok], g_sec[ok])[0, 1]),
                   "median_true_drop": float(np.median(t_sec[ok])), "median_genco_drop": float(np.median(g_sec[ok]))})
TB = pd.DataFrame(rows_b); TB["genco / true"] = TB.genco / TB.true
GB = pd.DataFrame(glob_b)
TB.to_csv(Path(OUT_DIR) / "load_growth_targets.csv", index=False)
GB.to_csv(Path(OUT_DIR) / "load_growth_all_buses.csv", index=False)
print(f"(b) {len(GRAD_KEYS)} scenarios in {time.time() - t0:.0f} s. Linear vs AC truth agree to "
      f"{np.nanmedian(np.abs(TB.true_linear - TB.true) / TB.true.abs()):.1%} (median).")
SUM_B = TB.groupby(["group", "hard"]).agg(pairs=("target", "size"), true_response=("true", "median"),
                                         genco_response=("genco", "median"), ratio=("genco / true", "median"))
print("Voltage change per unit of load growth (p.u.); ratio = GENCO / true, medians:")
display(SUM_B)
display(GB.groupby("hard")[["slope_all_pq", "corr_all_pq", "median_true_drop", "median_genco_drop"]].median())

## 7. (c) Generator voltage set-points

In [ ]:
rows_c = []
t0 = time.time()
for key in J2_KEYS:
    d = DICTS[key]; T = TrueResponses(d)
    pq, pv = PQM[key], PVM[key]
    tg = [int(t) for t in TARGETS if pq[int(t)]]
    H = hop_distances(d, tg)
    gens = {t: [int(g) for g in nearest(H[i], pv, N_SET)] for i, t in enumerate(tg)}
    allg = sorted({g for v in gens.values() for g in v})
    graphs = []
    for g in allg:
        for sgn in (1, -1):
            graphs.append(perturbed(key, lambda x, g=g, sgn=sgn: x.__setitem__((g, VM_H), x[g, VM_H] + sgn * DV_PU)))
    vms = genco_vm(graphs)
    for n, g in enumerate(allg):
        g_sec = (vms[2 * n] - vms[2 * n + 1]) / (2 * DV_PU)
        t_lin = T.setpoint(g)
        for i, t in enumerate(tg):
            if g in gens[t]:
                rows_c.append({"k": key[0], "scenario": key[1], "hard": HARD[key], "target": t, "group": GROUP[t], "generator_bus": g,
                               "hops": float(H[i][g]), "genco": float(g_sec[t]), "true": float(t_lin[t])})
TC = pd.DataFrame(rows_c)
TC["genco / true"] = TC.genco / TC.true.where(TC.true.abs() > 1e-6)
TC.to_csv(Path(OUT_DIR) / "setpoints.csv", index=False)
key0 = J2_KEYS[0]; T0 = TrueResponses(DICTS[key0]); g0 = int(np.nonzero(PVM[key0])[0][0])
v_ac, v0 = T0.resolve(dV=(g0, DV_PU)), T0.resolve()
pq0 = np.nonzero(PQM[key0])[0][:3]
print(f"(c) {len(TC)} target-generator pairs in {time.time() - t0:.0f} s. AC check: linear "
      f"{np.round(T0.setpoint(g0)[pq0], 4)} vs re-solved {np.round((v_ac[pq0] - v0[pq0]) / DV_PU, 4)}")
SUM_C = TC.groupby(["group", "hard"]).agg(pairs=("target", "size"), true_response=("true", "median"),
                                         genco_response=("genco", "median"), ratio=("genco / true", "median"))
print("Voltage change at the target per p.u. set-point change at its nearest generators; ratio = GENCO / true, medians:")
display(SUM_C)

## 8. Figures

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
ax = axes[0]
classes = ["own load", "near (1-2 lines)", "mid (3-5 lines), truth's largest", "mid (3-5 lines), GENCO's largest", "far (6+ lines), GENCO's largest"]
for off, grp_ in zip([-0.2, 0.2], ["wrong pattern", "random"]):
    data = [TA[(TA.group == grp_) & (TA["class"] == c)]["secant / true"].dropna().to_numpy() for c in classes]
    bp_ = ax.boxplot([x if len(x) else [np.nan] for x in data], positions=np.arange(len(classes)) + off, widths=0.35, patch_artist=True,
                     showfliers=False)
    for p in bp_["boxes"]:
        p.set_facecolor("#e07b20" if grp_ == "wrong pattern" else "#1f5fa8"); p.set_alpha(0.6)
ax.set_yscale("log"); ax.axhline(1, color="k", lw=1)
ax.set_xticks(range(len(classes))); ax.set_xticklabels([c.replace(", ", "\n") for c in classes], fontsize=8)
ax.set_ylabel(f"|GENCO response| / |true response| ({DQ_MVAR} Mvar)"); ax.set_title("(a) one load at a time: orange = wrong-pattern buses")
ax = axes[1]
for grp_, col_ in [("random", "#1f5fa8"), ("often missed, right pattern", "#2e8b57"), ("wrong pattern", "#e07b20")]:
    g_ = TB[TB.group == grp_]
    ax.scatter(g_.true, g_.genco, s=8, alpha=0.5, color=col_, label=grp_)
lo_, hi_ = np.nanmin(TB[["true", "genco"]].to_numpy()), np.nanmax(TB[["true", "genco"]].to_numpy())
ax.plot([lo_, hi_], [lo_, hi_], "k--", lw=1); ax.legend(frameon=False, fontsize=8)
ax.set_xlabel("true voltage change per unit load growth"); ax.set_ylabel("GENCO's"); ax.set_title("(b) all loads grow together")
ax = axes[2]
for grp_, col_ in [("random", "#1f5fa8"), ("often missed, right pattern", "#2e8b57"), ("wrong pattern", "#e07b20")]:
    g_ = TC[TC.group == grp_]
    ax.scatter(g_.true, g_.genco, s=10, alpha=0.6, color=col_, label=grp_)
ax.plot([0, 1], [0, 1], "k--", lw=1); ax.legend(frameon=False, fontsize=8)
ax.set_xlabel("true change at the target per p.u. set-point change"); ax.set_ylabel("GENCO's"); ax.set_title("(c) nearest generator set-points")
for a in axes:
    a.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(Path(OUT_DIR) / "fig_realistic_changes.png", dpi=150); plt.show()

## 9. Summary and output files

In [ ]:
med = lambda df, cond, col: float(df.loc[cond, col].median()) if cond.any() else np.nan
mid_w = med(TA, (TA.group == "wrong pattern") & (TA["class"] == "mid (3-5 lines), GENCO's largest"), "secant / true")
mid_r = med(TA, (TA.group == "random") & (TA["class"] == "mid (3-5 lines), GENCO's largest"), "secant / true")
agree_w = med(TA, (TA.group == "wrong pattern") & (TA["class"] == "mid (3-5 lines), GENCO's largest"), "secant / autograd")
grow_w = med(TB, TB.group == "wrong pattern", "genco / true"); grow_r = med(TB, TB.group == "random", "genco / true")
set_w = med(TC, TC.group == "wrong pattern", "genco / true"); set_r = med(TC, TC.group == "random", "genco / true")
MID_CONFIRMED = bool(mid_w >= 3 and 0.5 <= agree_w <= 2 and mid_w >= 2 * mid_r)
GROWTH_UNDER = bool(grow_w <= 0.8 and 0.9 <= grow_r <= 1.1)
SETPOINT_UNDER = bool(set_w <= 0.8 and 0.9 <= set_r <= 1.1)
print(f"(a) mid-distance loads, GENCO's largest: secant/true {mid_w:.2f} on wrong-pattern buses vs {mid_r:.2f} on random buses; "
      f"secant/autograd {agree_w:.2f}  ->  over-reaction confirmed at realistic size: {MID_CONFIRMED}")
print(f"(b) load growth, GENCO/true: wrong-pattern buses {grow_w:.2f}, random buses {grow_r:.2f}  ->  under-reacts where it compresses: {GROWTH_UNDER}")
print(f"(c) set-points, GENCO/true: wrong-pattern buses {set_w:.2f}, random buses {set_r:.2f}  ->  under-reacts where it compresses: {SETPOINT_UNDER}")
summary = {"settings": {"DQ_MVAR": DQ_MVAR, "GROWTH": GROWTH, "DV_PU": DV_PU, "N_NEAR": N_NEAR, "N_MID": N_MID, "N_FAR": N_FAR,
                        "N_GENQ": N_GENQ, "N_SET": N_SET, "J2_HARD": J2_HARD, "J2_NORMAL": J2_NORMAL, "SEED": SEED},
           "targets": TARGETS.tolist(), "a_targets": A_TARGETS.tolist(), "j2_scenarios": len(J2_KEYS), "b_scenarios": len(GRAD_KEYS),
           "mid_secant_over_true_wrong": mid_w, "mid_secant_over_true_random": mid_r, "mid_secant_over_autograd_wrong": agree_w,
           "growth_ratio_wrong": grow_w, "growth_ratio_random": grow_r, "setpoint_ratio_wrong": set_w, "setpoint_ratio_random": set_r,
           "MID_CONFIRMED": MID_CONFIRMED, "GROWTH_UNDER": GROWTH_UNDER, "SETPOINT_UNDER": SETPOINT_UNDER,
           "device": str(DEVICE), "versions": {"python": platform.python_version(), "torch": torch.__version__}}
json.dump(summary, open(Path(OUT_DIR) / "summary.json", "w"), indent=1, default=str)
zip_path = Path(OUT_DIR).with_suffix(".zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(Path(OUT_DIR).rglob("*")):
        if p.is_file():
            zf.write(p, arcname=str(p))
print(f"Saved {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB).")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass